# Hybrid Lorenz 96 path: burn-in + true solver vs DM

1. **Burn-in** — integrate the true Lorenz 96 solver until spin-up.
2. **Fork** — same post-burn-in state for two rollouts:
   - continue with the **true L96 solver**;
   - continue with **conditional diffusion** (autoregressive next-step prediction).
3. **Compare** — RMSE, selected dimension time series, ring snapshots, animation.

Set `NETWORK_PKL`, `K`, and `HISTORY` to match training (`lorenz96_path_next`, 1D U-Net).

In [ ]:
# --- User configuration ---
import os

from project_utils.paths import PROJECT_ROOT, dataset_trajectory_dir, CKPT_DIR

L96_CONFIG = os.path.join(PROJECT_ROOT, "configs/turbulence/lorenz96_default.yaml")
DATA_ROOT = "LORENZ96"
TRAJECTORY_SUBDIR = "full_path"

# Path to trained checkpoint (network-snapshot-*.pkl from train_cond.py)
# NETWORK_PKL = os.path.join(PROJECT_ROOT, "outputs", "00000-lorenz96_path_next-l96_unet1d_tiny-gpus1-batch256-fp32", "network-snapshot-000002.pkl")
NETWORK_PKL = "/home/neil/glocal-condiff-turb/outputs/00003-lorenz96_path_next-l96_unet1d_tiny-gpus1-batch256-fp32/network-snapshot-002000.pkl"


DEVICE = "cuda:0" if __import__("torch").cuda.is_available() else "cpu"
SEED = 0

K = 1000  # must match generator + checkpoint img_res_h
T_BURN_IN = 2000
T_ROLLOUT = 500  # comparison steps after burn-in
OUTPUT_INTERVAL = 0.01  # model time between saved frames (match generator YAML)

HISTORY = 1
VAR_SELECT = [0]

NUM_STEPS = 18  # EDM sampler steps
SEED_DM = 0

# Dimensions to plot in time-series comparison
DIM_INDICES = [0, 100, 500, 999]

In [ ]:
import pickle
import time

import dnnlib
import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.auto import tqdm

from project_utils.normalization import (
    denormalize_tensor,
    load_norm_stats,
    normalize_tensor,
    resolve_norm_stats_path,
)
from project_utils.yaml_simple import load_simple_yaml
from turbulence.lorenz96 import Lorenz96Params, Lorenz96Solver

device = torch.device(DEVICE)
trajectory_dir = dataset_trajectory_dir(DATA_ROOT, TRAJECTORY_SUBDIR)
norm_stats_path = resolve_norm_stats_path(trajectory_dir, None)
print("Trajectory dir:", trajectory_dir)
print("norm_stats:", norm_stats_path)
print("device:", device)

In [ ]:
def edm_sampler(
    net,
    latents,
    conditions,
    class_labels=None,
    randn_like=torch.randn_like,
    num_steps=18,
    sigma_min=0.002,
    sigma_max=80,
    rho=7,
    S_churn=0,
    S_min=0,
    S_max=float("inf"),
    S_noise=1,
):
    sigma_min = max(sigma_min, net.sigma_min)
    sigma_max = min(sigma_max, net.sigma_max)

    step_indices = torch.arange(num_steps, dtype=torch.float64, device=latents.device)
    t_steps = (
        sigma_max ** (1 / rho)
        + step_indices / (num_steps - 1) * (sigma_min ** (1 / rho) - sigma_max ** (1 / rho))
    ) ** rho
    t_steps = torch.cat([net.round_sigma(t_steps), torch.zeros_like(t_steps[:1])])

    x_next = latents.to(torch.float64) * t_steps[0]
    for i, (t_cur, t_next) in enumerate(zip(t_steps[:-1], t_steps[1:])):
        x_cur = x_next
        gamma = min(S_churn / num_steps, np.sqrt(2) - 1) if S_min <= t_cur <= S_max else 0
        t_hat = net.round_sigma(t_cur + gamma * t_cur)
        x_hat = x_cur + (t_hat**2 - t_cur**2).sqrt() * S_noise * randn_like(x_cur)

        denoised = net(x_hat, t_hat, conditions, class_labels).to(torch.float64)
        d_cur = (x_hat - denoised) / t_hat
        x_next = x_hat + (t_next - t_hat) * d_cur

        if i < num_steps - 1:
            denoised = net(x_next, t_next, conditions, class_labels).to(torch.float64)
            d_prime = (x_next - denoised) / t_next
            x_next = x_hat + (t_next - t_hat) * (0.5 * d_cur + 0.5 * d_prime)
    return x_next


class StackedRandomGenerator:
    def __init__(self, device, seeds):
        self.generators = [
            torch.Generator(device).manual_seed(int(s) % (1 << 32)) for s in seeds
        ]

    def randn(self, size, **kwargs):
        return torch.stack(
            [torch.randn(size[1:], generator=g, **kwargs) for g in self.generators]
        )

    def randn_like(self, input):
        return self.randn(input.shape, dtype=input.dtype, layout=input.layout, device=input.device)


def frame_tensor(x: torch.Tensor, k: int) -> torch.Tensor:
    """Map state (K,) to dataloader layout (1, K, 1)."""
    return x.reshape(1, k, 1)


def make_initial_condition(k: int, F: float, *, device, dtype: torch.dtype) -> torch.Tensor:
    x = torch.full((k,), float(F), device=device, dtype=dtype)
    x[0] = x[0] + torch.tensor(0.01, device=device, dtype=dtype)
    return x


def build_condition(frames_phys, history: int, stats: dict) -> torch.Tensor:
    """Match DatasetPathCond: stack last `history` normalized frames -> (1, history*C, K, 1)."""
    recent = frames_phys[-history:]
    normed = [normalize_tensor(f, stats) for f in recent]
    block = torch.stack(normed, dim=0)  # (history, C, K, 1)
    C, H, W = block.shape[1], block.shape[2], block.shape[3]
    cond = block.reshape(history * C, H, W)
    return cond.unsqueeze(0)


def to_seq(traj: torch.Tensor) -> torch.Tensor:
    """(T, 1, K, 1) -> (T, K) for plotting."""
    return traj[:, 0, :, 0]

## Phase 1: Lorenz 96 solver burn-in

Integrate the true model for `T_BURN_IN` saved frames (same spacing as `generate_fields.py`), then keep the state for the DM fork.

In [ ]:
cfg = load_simple_yaml(L96_CONFIG)
l96_cfg = dict(cfg.get("lorenz96", {}))
l96_cfg["device"] = str(device)

params = Lorenz96Params(K=K, **l96_cfg)
params.validate()
timesteps_per_output = max(1, int(OUTPUT_INTERVAL / params.dt))

dtype = torch.float32 if params.precision == "single" else torch.float64
x0 = make_initial_condition(K, params.F, device=device, dtype=dtype)
solver = Lorenz96Solver(x0, **params.to_core_kwargs())

path_burn = [frame_tensor(solver.fields()["x"].detach().cpu(), K)]
for _ in tqdm(range(T_BURN_IN), desc="L96 burn-in"):
    solver.step(timesteps_per_output)
    frame = frame_tensor(solver.fields()["x"].detach().cpu(), K)
    if not torch.isfinite(frame).all():
        raise RuntimeError("Non-finite values during burn-in")
    path_burn.append(frame)

state_after_burn = {k: v.clone() for k, v in solver.get_state().items()}
path_burn = torch.stack(path_burn, dim=0)  # (T_burn+1, 1, K, 1)

print(f"Burn-in: {path_burn.shape[0]} frames")
print(f"Last frame min/max: {path_burn[-1].min():.4g} / {path_burn[-1].max():.4g}")

## Phase 2: Load trained conditional diffusion model

In [ ]:
if not os.path.isfile(NETWORK_PKL):
    raise FileNotFoundError(
        f"Set NETWORK_PKL to your training snapshot. Not found: {NETWORK_PKL}\n"
        "Train with: torchrun scripts/train/train_cond.py runs=lorenz96_path_next"
    )

norm_stats = load_norm_stats(norm_stats_path)
print(f"Loaded norm_stats: vmin={norm_stats['vmin']:.6g}, vmax={norm_stats['vmax']:.6g}")

print(f"Loading network from {NETWORK_PKL} ...")
with dnnlib.util.open_url(NETWORK_PKL, verbose=True) as f:
    ckpt = pickle.load(f)
net = ckpt["ema"].to(device).eval()

n_target = len(VAR_SELECT)
n_cond = HISTORY * len(VAR_SELECT)
print(f"Model: {net.model_name}, target_ch={net.img_input_channels}, cond_ch={net.img_cond_channels}")
print(f"Resolution: H={net.img_res_h}, W={net.img_res_w}")
assert net.img_input_channels == n_target, "VAR_SELECT / checkpoint mismatch"
assert net.img_cond_channels == n_cond, "HISTORY / checkpoint mismatch"
assert net.img_res_h == K, f"K={K} must match trained seq length img_res_h={net.img_res_h}"
assert net.img_res_w == 1, "Expected img_res_w=1 for 1D L96 layout"

## Phase 3: Parallel rollout after burn-in

From the **same** post-burn-in state:
- **True solver** — continue integrating L96 for `T_ROLLOUT` saved frames.
- **DM** — autoregressive conditional diffusion for `T_ROLLOUT` frames.

In [ ]:
if path_burn.shape[0] < HISTORY:
    raise ValueError(f"Need at least HISTORY={HISTORY} burn-in frames")

core_kw = params.to_core_kwargs()

# --- True solver branch (restore forked state) ---
x_fork = path_burn[-1][0, :, 0].to(device=device, dtype=dtype)
solver_cont = Lorenz96Solver(x_fork, **core_kw)
solver_cont.set_state(state_after_burn)

rollout_solver = []
for _ in tqdm(range(T_ROLLOUT), desc="True solver rollout"):
    solver_cont.step(timesteps_per_output)
    frame = frame_tensor(solver_cont.fields()["x"].detach().cpu(), K)
    if not torch.isfinite(frame).all():
        raise RuntimeError("Non-finite values during solver rollout")
    rollout_solver.append(frame)
path_solver = torch.stack(rollout_solver, dim=0)  # (T_ROLLOUT, 1, K, 1)

# --- DM branch (same initial history as end of burn-in) ---
path_dm_hist = [path_burn[i].clone() for i in range(path_burn.shape[0])]
rnd = StackedRandomGenerator(device, [SEED_DM])

rollout_dm = []
t0 = time.time()
for step in tqdm(range(T_ROLLOUT), desc="DM rollout"):
    cond = build_condition(path_dm_hist, HISTORY, norm_stats).to(device)
    latents = rnd.randn([1, n_target, K, 1], device=device)
    with torch.no_grad():
        sample_norm = edm_sampler(net, latents, cond, num_steps=NUM_STEPS).to(torch.float32)
    next_phys = denormalize_tensor(sample_norm[0].cpu(), norm_stats)
    if not torch.isfinite(next_phys).all():
        raise RuntimeError(f"Non-finite DM sample at step {step}")
    rollout_dm.append(next_phys)
    path_dm_hist.append(next_phys)

path_dm = torch.stack(rollout_dm, dim=0)  # (T_ROLLOUT, 1, K, 1)
print(f"DM rollout done in {time.time() - t0:.1f}s")
print("Shapes:", path_solver.shape, path_dm.shape)

## Phase 4: Compare true solver vs DM

In [ ]:
solver_seq = to_seq(path_solver)  # (T, K)
dm_seq = to_seq(path_dm)

diff = solver_seq - dm_seq
rmse = diff.pow(2).mean(dim=1).sqrt()  # (T,)
rel_rmse = rmse / solver_seq.pow(2).mean(dim=1).sqrt().clamp_min(1e-8)

steps = np.arange(T_ROLLOUT)
fig, axs = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
axs[0].plot(steps, rmse.numpy(), color="C0")
axs[0].set_ylabel("RMSE (physical x)")
axs[0].set_title("True solver vs DM — error after shared burn-in")
axs[0].grid(True, alpha=0.3)
axs[1].plot(steps, rel_rmse.numpy(), color="C1")
axs[1].set_ylabel("relative RMSE")
axs[1].set_xlabel("rollout step k (after burn-in)")
axs[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Mean RMSE over rollout: {rmse.mean().item():.4g}")
print(f"Final-step RMSE: {rmse[-1].item():.4g}")

In [ ]:
# Time series: selected dimensions
valid_dims = [d for d in DIM_INDICES if 0 <= d < K]
fig, axs = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

for d in valid_dims:
    axs[0].plot(solver_seq[:, d].numpy(), label=f"x[{d}]", linewidth=1.0)
axs[0].set_ylabel("x (true solver)")
axs[0].set_title("Selected dimensions — true solver")
axs[0].legend(loc="upper right", ncol=2, fontsize=8)
axs[0].grid(True, alpha=0.3)

for d in valid_dims:
    axs[1].plot(dm_seq[:, d].numpy(), label=f"x[{d}]", linewidth=1.0)
axs[1].set_ylabel("x (DM)")
axs[1].set_title("Selected dimensions — diffusion model")
axs[1].set_xlabel("rollout step k")
axs[1].legend(loc="upper right", ncol=2, fontsize=8)
axs[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Overlay one dimension: true vs DM
compare_dim = valid_dims[0]
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(solver_seq[:, compare_dim].numpy(), label="true solver", linewidth=1.2)
ax.plot(dm_seq[:, compare_dim].numpy(), label="DM", linewidth=1.2, alpha=0.85)
ax.set_xlabel("rollout step k")
ax.set_ylabel(f"x[{compare_dim}]")
ax.set_title(f"Dimension {compare_dim}: true solver vs DM")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Ring snapshot at a selected rollout step
k_show = min(20, T_ROLLOUT - 1)
x_axis = np.arange(K)

fig, axs = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
axs[0].plot(x_axis, solver_seq[k_show].numpy(), color="C0")
axs[0].set_ylabel("x")
axs[0].set_title(f"True solver, k={k_show}")
axs[0].grid(True, alpha=0.3)
axs[1].plot(x_axis, dm_seq[k_show].numpy(), color="C1")
axs[1].set_ylabel("x")
axs[1].set_title(f"DM, k={k_show}")
axs[1].set_xlabel("dimension index i")
axs[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Space-time heatmap: true vs DM vs error
TIME_STRIDE = max(1, T_ROLLOUT // 200)
DIM_STRIDE = max(1, K // 500)

fig, axs = plt.subplots(1, 3, figsize=(14, 4))
for ax, data, title in zip(
    axs,
    [solver_seq.numpy(), dm_seq.numpy(), diff.abs().numpy()],
    ["True solver", "DM", "|error|"],
):
    im = ax.imshow(
        data[::TIME_STRIDE, ::DIM_STRIDE].T,
        aspect="auto",
        origin="lower",
        cmap="RdBu_r" if title != "|error|" else "magma",
    )
    ax.set_title(title)
    ax.set_xlabel("rollout step")
    ax.set_ylabel("dimension")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

## Animate ring comparison (post burn-in)

Top: **true L96 solver**. Bottom: **conditional diffusion**.

In [ ]:
from IPython.display import HTML
from matplotlib.animation import FuncAnimation

interval_ms = 80
max_frames = min(T_ROLLOUT, 150)
frame_ids = list(range(max_frames))
x_axis = np.arange(K)

stacked = torch.cat([solver_seq[:max_frames], dm_seq[:max_frames]], dim=0)
vmin = float(stacked.min())
vmax = float(stacked.max())

fig, axs = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
lines = []
for ax, label in zip(axs, ["true L96 solver", "conditional diffusion (DM)"]):
    (line,) = ax.plot(x_axis, solver_seq[0].numpy(), linewidth=1.0)
    lines.append(line)
    ax.set_ylim(vmin, vmax)
    ax.set_ylabel("x")
    ax.set_title(label)
    ax.grid(True, alpha=0.3)
axs[-1].set_xlabel("dimension index i")
title = fig.suptitle("")


def update(frame_idx):
    k = frame_ids[frame_idx]
    lines[0].set_ydata(solver_seq[k].numpy())
    lines[1].set_ydata(dm_seq[k].numpy())
    title.set_text(f"rollout step k = {k} (after burn-in)")
    return [*lines, title]


anim = FuncAnimation(
    fig,
    update,
    frames=len(frame_ids),
    interval=interval_ms,
    blit=True,
    repeat=True,
)
plt.close(fig)
HTML(anim.to_jshtml())

In [ ]:
# Optional: save comparison tensors
# SAVE_PATH = os.path.join(PROJECT_ROOT, "scripts/Lorenz96/hybrid_solver_dm_compare.pt")
# torch.save(
#     {
#         "path_burn": path_burn,
#         "path_solver": path_solver,
#         "path_dm": path_dm,
#         "rmse": rmse,
#         "meta": {
#             "K": K,
#             "T_burn_in": T_BURN_IN,
#             "T_rollout": T_ROLLOUT,
#             "history": HISTORY,
#             "network_pkl": NETWORK_PKL,
#             "l96_config": L96_CONFIG,
#         },
#     },
#     SAVE_PATH,
# )
# print("Saved:", SAVE_PATH)